# Build Sentinel-2 test dataset
Run the cell below to execute the same workflow previously provided as a Python script.

In [1]:
#!/usr/bin/env python3
"""Build a Sentinel-2 test dataset anchored to reference orthoimagery.

The script scans two Windows directories containing ortho TIFFs, extracts
approximate acquisition windows from the filenames/metadata, and fetches
matching Sentinel-2 chips clipped to the ortho footprint. Exported files
inherit the exact TIFF name so downstream notebooks can treat them as
one-to-one surrogates of the reference data.
"""
from __future__ import annotations

import argparse
import dataclasses
import logging
import re
from datetime import datetime, timedelta
from pathlib import Path
try:
    NOTEBOOK_ROOT = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_ROOT = Path.cwd()
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import ee
import geemap
import numpy as np
import fiona
import rasterio
from rasterio.errors import RasterioIOError
from rasterio.features import rasterize
from rasterio.warp import transform_geom
from rasterio.warp import transform_bounds

LOGGER = logging.getLogger("build_test_dataset")
DEFAULT_ORTHO_DIRS = (
    "F:/Snezisca_za_FERI/Skedenj/Ortofoto",
    "F:/Snezisca_za_FERI/Martuljska_skupina/Ortofoto",
)
DEFAULT_SHAPE_DIRS = {
    "skedenj": "F:/Snezisca_za_FERI/Skedenj/SHP-ji",
    "martuljska_skupina": "F:/Snezisca_za_FERI/Martuljska_skupina/SHP-ji",
}
DEFAULT_DW_LABEL_DIR = "test_dataset/all_labels"
DEFAULT_DW_PROB_DIR = "test_dataset/all_labels_probabilities"
DEFAULT_SNOW_LABEL_DIR = "test_dataset/snow_labels"
DEFAULT_GT_LABEL_DIR = "test_dataset/given_gt_snow_labels"
DYNAMIC_WORLD_INITIAL_WINDOW_DAYS = 10
DYNAMIC_WORLD_MAX_WINDOW_DAYS = 30
DYNAMIC_WORLD_WINDOW_STEP = 10
SNOW_CLASS_ID = 8
SNOW_MIN_PROB = 0.51
DW_PROBABILITY_BANDS = (
    "water",
    "trees",
    "grass",
    "flooded_vegetation",
    "crops",
    "shrub_and_scrub",
    "built",
    "bare",
    "snow_and_ice",
)
S2_COLLECTION = "COPERNICUS/S2_HARMONIZED"
S2_BANDS = ("B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B9", "B10", "B11", "B12")
YEAR_REGEX = re.compile(r"(19|20)\\d{2}")
DATE_TAG_FORMAT = "%Y:%m:%d %H:%M:%S"
MIN_S2_DATE = datetime(2015, 6, 23)
DATE_BY_SITE_YEAR: Dict[str, Dict[int, datetime]] = {
    "skedenj": {
        2006: datetime(2006, 8, 5),
        2015: datetime(2015, 8, 22),
        2017: datetime(2017, 8, 6),
        2020: datetime(2020, 7, 31),
        2023: datetime(2023, 9, 9),
    },
    "martuljska_skupina": {
        2006: datetime(2006, 7, 22),
        2011: datetime(2011, 8, 11),
        2015: datetime(2015, 7, 5),
        2017: datetime(2017, 8, 24),
        2020: datetime(2020, 7, 28),
        2023: datetime(2023, 8, 21),
    },
}
NOTEBOOK_ARGS: List[str] = []  # edit this list to pass CLI-style arguments inside Jupyter


@dataclasses.dataclass
class OrthoTile:
    """Metadata extracted from an ortho TIFF on the F: drive."""

    source_path: Path
    name: str
    lonlat_bounds: Tuple[float, float, float, float]
    crs: str
    reference_date: datetime
    start_date: datetime
    end_date: datetime

    @property
    def region(self) -> ee.Geometry:
        return ee.Geometry.Rectangle(self.lonlat_bounds)


def normalize_windows_path(path_str: str) -> Path:
    """Convert a Windows-style drive path into something usable inside WSL."""

    candidate = Path(path_str)
    if candidate.exists():
        return candidate
    if ":" not in path_str:
        return candidate
    drive, remainder = path_str.split(":", 1)
    remainder = remainder.strip("\\/")
    translated = Path("/mnt") / drive.lower() / remainder.replace("\\", "/")
    return translated


def _shape_dir_for_site(site_key: str) -> Optional[Path]:
    shape_dir = DEFAULT_SHAPE_DIRS.get(site_key)
    if not shape_dir:
        return None
    normalized = normalize_windows_path(shape_dir)
    return normalized if normalized.exists() else None


def _shapefile_for_tile(tile: OrthoTile) -> Optional[Path]:
    site_key = _site_key_from_path(tile.source_path)
    if not site_key:
        return None
    shape_dir = _shape_dir_for_site(site_key)
    if not shape_dir:
        return None
    year = _year_from_name(tile.source_path.stem)
    if year is None:
        return None
    if site_key == "skedenj":
        candidate = shape_dir / f"prisojnik_3D_{year}.shp"
    else:
        candidate = shape_dir / f"Martuljska_{year}.shp"
    return candidate if candidate.exists() else None


def export_dynamic_world(
    tile: OrthoTile,
    destination: Path,
    dw_label_dir: Path,
    dw_prob_dir: Path,
    snow_label_dir: Path,
    overwrite: bool = False,
) -> None:
    dw_label_dir.mkdir(parents=True, exist_ok=True)
    dw_prob_dir.mkdir(parents=True, exist_ok=True)
    snow_label_dir.mkdir(parents=True, exist_ok=True)
    dw_label_path = dw_label_dir / destination.name
    dw_prob_path = dw_prob_dir / destination.name
    snow_path = snow_label_dir / destination.name
    if not overwrite and dw_label_path.exists() and dw_prob_path.exists() and snow_path.exists():
        return
    label_img, prob_img = fetch_dynamic_world_stack(tile.region, tile.reference_date)
    combined_dw = label_img.addBands(prob_img)
    snow_img = derive_binary_snow_mask(combined_dw, SNOW_MIN_PROB)
    geemap.ee_export_image(
        label_img,
        filename=str(dw_label_path),
        region=tile.region,
        scale=10,
        crs="EPSG:4326",
        file_per_band=False,
    )
    geemap.ee_export_image(
        prob_img,
        filename=str(dw_prob_path),
        region=tile.region,
        scale=10,
        crs="EPSG:4326",
        file_per_band=False,
    )
    geemap.ee_export_image(
        snow_img,
        filename=str(snow_path),
        region=tile.region,
        scale=10,
        crs="EPSG:4326",
        file_per_band=False,
    )


def _write_snow_label(tile: OrthoTile, destination: Path, labels_dir: Path) -> Path:
    labels_dir.mkdir(parents=True, exist_ok=True)
    label_path = labels_dir / destination.name
    with rasterio.open(destination) as dataset:
        profile = dataset.profile
        height = dataset.height
        width = dataset.width
        transform = dataset.transform
        dst_crs = dataset.crs
    mask = np.zeros((height, width), dtype=np.uint8)
    shapefile = _shapefile_for_tile(tile)
    if shapefile is None:
        LOGGER.warning("No shapefile found for %s; writing empty mask", tile.name)
    else:
        with fiona.open(shapefile) as src:
            src_crs = src.crs
            geometries = [feature["geometry"] for feature in src]
        if src_crs and dst_crs and src_crs != dst_crs:
            geometries = [
                transform_geom(src_crs, dst_crs, geom) for geom in geometries
            ]
        if geometries:
            mask = rasterize(
                geometries,
                out_shape=(height, width),
                transform=transform,
                fill=0,
                default_value=1,
                dtype=np.uint8,
            )
    profile.update(count=1, dtype=rasterio.uint8, nodata=0)
    with rasterio.open(label_path, "w", **profile) as dst:
        dst.write(mask, 1)
    return label_path


def iter_tiff_paths(root: Path) -> Iterable[Path]:
    for path in sorted(root.glob("*.tif")):
        if path.name.endswith(".aux.xml"):
            continue
        yield path


def _site_key_from_path(path: Path) -> Optional[str]:
    lowered = [part.lower() for part in path.parts]
    if "skedenj" in lowered:
        return "skedenj"
    if "martuljska_skupina" in lowered or "martuljska skupina" in lowered:
        return "martuljska_skupina"
    return None


def _date_from_lookup(tile_path: Path) -> Optional[datetime]:
    site_key = _site_key_from_path(tile_path)
    if not site_key:
        LOGGER.warning("Unable to infer site from %s", tile_path)
        return None
    year = _year_from_name(tile_path.stem)
    if year is None:
        return None
    site_dates = DATE_BY_SITE_YEAR.get(site_key, {})
    date = site_dates.get(year)
    if date is None:
        LOGGER.warning("No reference date for %s year %s", site_key, year)
        return None
    return date


def _date_from_tags(tags: Dict[str, str]) -> Optional[datetime]:
    stamp = tags.get("TIFFTAG_DATETIME")
    if not stamp:
        return None
    try:
        return datetime.strptime(stamp, DATE_TAG_FORMAT)
    except ValueError:
        LOGGER.warning("Unable to parse TIFFTAG_DATETIME='%s'", stamp)
        return None


def _year_from_name(name: str) -> Optional[int]:
    parts = name.split("_")
    if len(parts) > 1:
        try:
            return int(parts[1])
        except ValueError:
            LOGGER.warning("Unable to parse year from filename %s", name)
    match = YEAR_REGEX.search(name)
    return int(match.group()) if match else None


def read_ortho_tiles(directories: Sequence[str]) -> List[OrthoTile]:
    tiles: List[OrthoTile] = []
    for directory in directories:
        normalized = normalize_windows_path(directory)
        if not normalized.exists():
            LOGGER.warning("Directory %s not found", directory)
            continue
        for tiff_path in iter_tiff_paths(normalized):
            try:
                with rasterio.open(tiff_path) as dataset:
                    bounds = transform_bounds(
                        dataset.crs, "EPSG:4326", *dataset.bounds, densify_pts=21
                    )
                    tags = dataset.tags()
            except RasterioIOError as exc:
                LOGGER.error("Unable to read %s: %s", tiff_path, exc)
                continue
            timestamp = _date_from_tags(tags)
            if timestamp:
                reference_date = timestamp
                start_date = reference_date - timedelta(days=20)
                end_date = reference_date + timedelta(days=20)
            else:
                reference_date = _date_from_lookup(tiff_path)
                if reference_date is None:
                    continue
                start_date = reference_date - timedelta(days=20)
                end_date = reference_date + timedelta(days=20)
            tiles.append(
                OrthoTile(
                    source_path=tiff_path,
                    name=tiff_path.name,
                    lonlat_bounds=bounds,
                    crs=str(dataset.crs),
                    reference_date=reference_date,
                    start_date=start_date,
                    end_date=end_date,
                )
            )
    tiles.sort(key=lambda tile: tile.source_path)
    LOGGER.info("Discovered %d ortho tiles", len(tiles))
    return tiles


def initialize_earth_engine() -> None:
    try:
        ee.Initialize()
    except Exception:
        LOGGER.info("Authenticating Google Earth Engine interactively…")
        ee.Authenticate()
        ee.Initialize()


def fetch_dynamic_world_stack(
    aoi: ee.Geometry,
    reference_day: datetime,
    initial_window: int = DYNAMIC_WORLD_INITIAL_WINDOW_DAYS,
    max_window: int = DYNAMIC_WORLD_MAX_WINDOW_DAYS,
    step: int = DYNAMIC_WORLD_WINDOW_STEP,
) -> tuple[ee.Image, ee.Image]:
    window_days = initial_window
    while window_days <= max_window:
        start = (reference_day - timedelta(days=window_days)).strftime("%Y-%m-%d")
        end = (reference_day + timedelta(days=window_days + 1)).strftime("%Y-%m-%d")
        collection = (
            ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
            .filterBounds(aoi)
            .filterDate(start, end)
        )
        count = collection.size().getInfo()
        if count > 0:
            label_image = collection.select("label").mode().toUint8().rename("label").clip(aoi)
            prob_image = collection.select(DW_PROBABILITY_BANDS).median().clip(aoi)
            return label_image, prob_image
        window_days += step
    raise RuntimeError("No Dynamic World scenes found for requested window")


def derive_binary_snow_mask(dw_image: ee.Image, min_prob: float) -> ee.Image:
    snow_label = dw_image.select("label").eq(SNOW_CLASS_ID)
    snow_prob = dw_image.select("snow_and_ice").gte(min_prob)
    return snow_label.Or(snow_prob).rename("snow_mask")


def add_valid_fraction(image: ee.Image, region: ee.Geometry) -> ee.Image:
    mask = image.select("B8").mask()
    stats = mask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=20,
        bestEffort=True,
        maxPixels=int(1e8),
    )
    return image.set("valid_fraction", stats.get("B8"))


def fetch_best_sentinel_tile(tile: OrthoTile) -> Optional[ee.Image]:
    if tile.end_date < MIN_S2_DATE:
        LOGGER.warning("Skipping %s; no Sentinel-2 coverage before %s", tile.name, MIN_S2_DATE.date())
        return None
    collection = (
        ee.ImageCollection(S2_COLLECTION)
        .filterBounds(tile.region)
        .filterDate(tile.start_date.strftime("%Y-%m-%d"), tile.end_date.strftime("%Y-%m-%d"))
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
        .map(lambda img: add_valid_fraction(img, tile.region))
        .filter(ee.Filter.gte("valid_fraction", 0.98))
        .sort("CLOUDY_PIXEL_PERCENTAGE")
    )
    if collection.size().getInfo() == 0:
        LOGGER.warning("No Sentinel-2 images for %s in date window", tile.name)
        return None
    image = ee.Image(collection.first()).select(S2_BANDS)
    return image.clip(tile.region)


def download_tile_image(
    tile: OrthoTile,
    output_dir: Path,
    labels_dir: Path,
    dw_label_dir: Path,
    dw_prob_dir: Path,
    snow_label_dir: Path,
    overwrite: bool = False,
) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(tile.name).stem
    suffix = Path(tile.name).suffix or ".tif"
    date_stamp = tile.reference_date.strftime("%d_%m_%Y")
    destination = output_dir / f"{stem}_{date_stamp}{suffix}"
    if destination.exists() and not overwrite:
        LOGGER.info("Skipping existing tile %s", destination)
        _write_snow_label(tile, destination, labels_dir)
        export_dynamic_world(tile, destination, dw_label_dir, dw_prob_dir, snow_label_dir, overwrite)
        return destination
    image = fetch_best_sentinel_tile(tile)
    if image is None:
        return destination
    LOGGER.info("Exporting %s -> %s", tile.name, destination)
    geemap.ee_export_image(
        image,
        filename=str(destination),
        region=tile.region,
        scale=10,
        crs="EPSG:4326",
        file_per_band=False,
    )
    _write_snow_label(tile, destination, labels_dir)
    export_dynamic_world(tile, destination, dw_label_dir, dw_prob_dir, snow_label_dir, overwrite)
    return destination


def build_test_dataset(
    directories: Sequence[str],
    output_dir: Path,
    labels_dir: Path,
    dw_label_dir: Path,
    dw_prob_dir: Path,
    snow_label_dir: Path,
    max_tiles: Optional[int] = None,
    overwrite: bool = False,
) -> list[Path]:
    initialize_earth_engine()
    tiles = read_ortho_tiles(directories)
    exported: list[Path] = []
    for idx, tile in enumerate(tiles):
        if max_tiles is not None and idx >= max_tiles:
            break
        try:
            exported_path = download_tile_image(
                tile, output_dir, labels_dir, dw_label_dir, dw_prob_dir, snow_label_dir, overwrite=overwrite
            )
        except Exception as exc:  # pragma: no cover - Earth Engine failures
            LOGGER.exception("Failed to export %s: %s", tile.name, exc)
            continue
        exported.append(exported_path)
    LOGGER.info("Exported %d Sentinel-2 chips", len(exported))
    return exported


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--ortho-dirs",
        nargs="*",
        default=DEFAULT_ORTHO_DIRS,
        help="Directories containing reference ortho TIFFs",
    )
    parser.add_argument(
        "--output-root",
        type=Path,
        default=NOTEBOOK_ROOT / "test_dataset" / "raw",
        help="Destination folder for Sentinel-2 chips",
    )
    parser.add_argument(
        "--dw-label-dir",
        type=Path,
        default=NOTEBOOK_ROOT / "test_dataset" / "all_labels",
        help="Destination folder for Dynamic World labels",
    )
    parser.add_argument(
        "--dw-prob-dir",
        type=Path,
        default=NOTEBOOK_ROOT / "test_dataset" / "all_labels_probabilities",
        help="Destination folder for Dynamic World probabilities",
    )
    parser.add_argument(
        "--snow-label-dir",
        type=Path,
        default=NOTEBOOK_ROOT / "test_dataset" / "snow_labels",
        help="Destination folder for derived snow masks",
    )
    parser.add_argument(
        "--labels-dir",
        type=Path,
        default=NOTEBOOK_ROOT / "test_dataset" / "given_gt_snow_labels",
        help="Destination folder for given ground-truth snow masks",
    )
    parser.add_argument(
        "--max-tiles",
        type=int,
        default=None,
        help="Limit the number of tiles exported during dry runs",
    )
    parser.add_argument(
        "--overwrite",
        action="store_true",
        help="Re-download files even if they already exist",
    )
    parser.add_argument(
        "--log-level",
        default="INFO",
        choices=("DEBUG", "INFO", "WARNING", "ERROR"),
        help="Logging verbosity",
    )
    return parser.parse_args(argv)


def main() -> None:
    args = parse_args(NOTEBOOK_ARGS or None)
    logging.basicConfig(level=getattr(logging, args.log_level))
    exported = build_test_dataset(
        directories=args.ortho_dirs,
        output_dir=args.output_root,
        labels_dir=args.labels_dir,
        dw_label_dir=args.dw_label_dir,
        dw_prob_dir=args.dw_prob_dir,
        snow_label_dir=args.snow_label_dir,
        max_tiles=args.max_tiles,
        overwrite=args.overwrite,
    )
    if not exported:
        LOGGER.warning("No Sentinel-2 tiles were exported. Check logs above.")


if __name__ == "__main__":
    import sys
    if any(arg.startswith("-f") for arg in sys.argv[1:]):
        sys.argv = [sys.argv[0]]
    main()

C:\Users\Domen\anaconda3\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
INFO:build_test_dataset:Discovered 11 ortho tiles
INFO:build_test_dataset:Exporting POF_2015.tif -> d:\analiza_snezisc2\2_download_test_data_and_create_test_dataset\test_dataset\raw\POF_2015_05_07_2015.tif


Generating URL ...
Please wait ...


Data downloaded to d:\analiza_snezisc2\2_download_test_data_and_create_test_dataset\test_dataset\raw\POF_2015_05_07_2015.tif
Generating URL ...
Please wait ...
Data downloaded to d:\analiza_snezisc2\2_download_test_data_and_create_test_dataset\test_dataset\all_labels\POF_2015_05_07_2015.tif
Generating URL ...
Please wait ...
Data downloaded to d:\analiza_snezisc2\2_download_test_data_and_create_test_dataset\test_dataset\all_labels_probabilities\POF_2015_05_07_2015.tif
Generating URL ...
Please wait ...
Data downloaded to d:\analiza_snezisc2\2_download_test_data_and_create_test_dataset\test_dataset\snow_labels\POF_2015_05_07_2015.tif


INFO:build_test_dataset:Exporting POF_2017.tif -> d:\analiza_snezisc2\2_download_test_data_and_create_test_dataset\test_dataset\raw\POF_2017_24_08_2017.tif


Generating URL ...
Please wait ...


...

INFO:build_test_dataset:Exporting POF_2023_skedenj.tif -> d:\analiza_snezisc2\2_download_test_data_and_create_test_dataset\test_dataset\raw\POF_2023_skedenj_09_09_2023.tif


Generating URL ...
Please wait ...


Data downloaded to d:\analiza_snezisc2\2_download_test_data_and_create_test_dataset\test_dataset\raw\POF_2023_skedenj_09_09_2023.tif
Generating URL ...
Please wait ...
Data downloaded to d:\analiza_snezisc2\2_download_test_data_and_create_test_dataset\test_dataset\all_labels\POF_2023_skedenj_09_09_2023.tif
Generating URL ...
Please wait ...
Data downloaded to d:\analiza_snezisc2\2_download_test_data_and_create_test_dataset\test_dataset\all_labels_probabilities\POF_2023_skedenj_09_09_2023.tif
Generating URL ...
Please wait ...


INFO:build_test_dataset:Exported 11 Sentinel-2 chips


Data downloaded to d:\analiza_snezisc2\2_download_test_data_and_create_test_dataset\test_dataset\snow_labels\POF_2023_skedenj_09_09_2023.tif
